<h1 align="center">Homework 4</h1>

**Assigned:** Session 7, November 7, 2026

**Due:** Session 9, November 21, 2026

**Covers:** Lectures 7 and 8

**Instructions:** This set is a single Jupyter notebook, distributed and submitted as a `.ipynb` file; nothing is submitted on paper. Each problem is a markdown cell followed by one or more code cells with a marked location for your work; do not change the markdown, the function signatures, or the final self-check cell. Use only the Python standard library and NumPy. Run every cell from a clean kernel before submitting, top to bottom, and submit the notebook with its output retained rather than cleared, so that a reader can see the results without re-executing anything. Part A can be attempted after Lecture 7 and Part B after Lecture 8. No question depends on Lecture 9.

**Conventions:** As in Homeworks 1 through 3. The quantum Fourier transform on $n$ qubits is $\mathrm{QFT}\ket{x} = 2^{-n/2}\sum_{y} e^{2\pi i xy/2^n}\ket{y}$, with $x$ and $y$ read as integers whose most significant bit is on the top wire. Bell states are $\Phi^+ = (\ket{00} + \ket{11})/\sqrt{2}$, $\Phi^- = (\ket{00} - \ket{11})/\sqrt{2}$, $\Psi^+ = (\ket{01} + \ket{10})/\sqrt{2}$, $\Psi^- = (\ket{01} - \ket{10})/\sqrt{2}$.

## Setup

Run this cell first. Do not modify it.

In [ ]:
import numpy as np
from math import gcd

k0 = np.array([1, 0], dtype=complex)
k1 = np.array([0, 1], dtype=complex)
plus = (k0 + k1) / np.sqrt(2)
minus = (k0 - k1) / np.sqrt(2)
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)


def kron(*factors):
    out = factors[0]
    for factor in factors[1:]:
        out = np.kron(out, factor)
    return out


def ket(bits):
    return kron(*[(k1 if b == "1" else k0) for b in bits])


def close(a, b, tol=1e-9):
    return np.allclose(a, b, atol=tol)


Phip = (ket("00") + ket("11")) / np.sqrt(2)
Phim = (ket("00") - ket("11")) / np.sqrt(2)
Psip = (ket("01") + ket("10")) / np.sqrt(2)
Psim = (ket("01") - ket("10")) / np.sqrt(2)


## Part A: Fourier methods, phase estimation, and order finding

Attempt after Lecture 7.

### Problem A1: the quantum Fourier transform, by matrix and by circuit

Complete `qft_matrix(n)` so it returns the $2^n \times 2^n$ matrix of $\mathrm{QFT}$ on $n$ qubits directly from the definition above, and `qft(state)` so it applies that matrix to a state vector. The self-check below computes $\mathrm{QFT}\ket{10}$ and $\mathrm{QFT}\ket{11}$ for $n = 2$ and confirms they are orthogonal, as any two images of orthogonal inputs under a unitary must be.

In [ ]:
def qft_matrix(n):
    # TODO: return the 2**n x 2**n matrix with entries 2**(-n/2) * exp(2j * pi * x * y / 2**n).
    raise NotImplementedError


def qft(state):
    # TODO: apply qft_matrix to state (infer n from the length of state).
    raise NotImplementedError


Self-check. Do not edit this cell.

In [ ]:
out_10 = qft(ket("10"))
out_11 = qft(ket("11"))
assert close(out_10, 0.5 * np.array([1, -1, 1, -1])), "QFT|10> should be (|00>-|01>+|10>-|11>)/2"
assert close(out_11, 0.5 * np.array([1, -1j, -1, 1j])), "QFT|11> should be (1,-i,-1,i)/2"
assert close(np.vdot(out_10, out_11), 0), "images of orthogonal inputs under a unitary must be orthogonal"
print("A1 passed.")


### Problem A2: phase estimation, exact and inexact

Let $U$ act on a single eigenstate $\ket{1}$ with $U\ket{1} = e^{2\pi i \varphi}\ket{1}$. With $t$ counting qubits, phase estimation puts each counting qubit through $H$, applies controlled-$U^{2^j}$ from counting qubit $j$ (with $j = t - 1$ on the top wire), applies $\mathrm{QFT}^{-1}$, and measures. Because the eigenstate is fixed, the counting register alone, before the inverse transform, is $2^{-t/2}\sum_{y=0}^{2^t - 1} e^{2\pi i \varphi y}\ket{y}$.

Complete `phase_estimation_probs(t, phi)` so it builds that register directly from the formula above (no controlled-$U$ circuit is required, since the target is fixed to be an eigenstate throughout) and returns the length-$2^t$ array of outcome probabilities after $\mathrm{QFT}^{-1}$.

In [ ]:
def phase_estimation_probs(t, phi):
    # TODO: build the counting register 2**(-t/2) * sum_y exp(2j*pi*phi*y) |y>, apply the inverse QFT,
    # and return the array of outcome probabilities (length 2**t).
    raise NotImplementedError


Self-check. At $t = 3$, $\varphi = 3/8$ has an exact 3-bit expansion, so the outcome $\ket{011} = \ket{3}$ occurs with certainty. At $t = 2$, $\varphi = 3/8$ has no exact 2-bit expansion; the two nearest estimates ($k = 1$ and $k = 2$) should each occur with probability $(2 + \sqrt{2})/8 \approx 0.427$, above the lecture's guaranteed bound $4/\pi^2 \approx 0.405$. Do not edit this cell.

In [ ]:
p3 = phase_estimation_probs(3, 3 / 8)
assert close(p3, np.eye(8)[3]), "t=3, phi=3/8 should return |011> with certainty"
p2 = phase_estimation_probs(2, 3 / 8)
expected = np.array([2 - np.sqrt(2), 2 + np.sqrt(2), 2 + np.sqrt(2), 2 - np.sqrt(2)]) / 8
assert close(p2, expected), "t=2, phi=3/8 probabilities should be (2-sqrt2)/8, (2+sqrt2)/8, (2+sqrt2)/8, (2-sqrt2)/8"
assert p2[1] > 4 / np.pi ** 2, "the nearest estimate's probability should exceed the lecture's bound 4/pi^2"
print("A2 passed.")


### Problem A3: order finding for $N = 21$

Complete `order(a, N)` so it returns the multiplicative order of `a` modulo `N` (the smallest $k \geq 1$ with $a^k \equiv 1 \pmod N$), without using any library beyond the standard one. Then use it, together with `math.gcd`, to carry out the classical reduction from an order to a factor of $21$ for $a = 2$ and to show why $a = 5$ yields nothing.

In [ ]:
def order(a, N):
    # TODO: return the smallest k >= 1 with pow(a, k, N) == 1.
    raise NotImplementedError


def factor_attempt(a, N):
    '''Return (r, g1, g2) where r = order(a, N), g1 = gcd(a**(r//2) - 1, N), g2 = gcd(a**(r//2) + 1, N),
    following the reduction from order finding to factoring. Assumes r is even; raise ValueError otherwise.'''
    # TODO
    raise NotImplementedError


Self-check. For $a = 2$, $N = 21$: order $6$, and the two gcds should be the factors $7$ and $3$. For $a = 5$, $N = 21$: the order is also $6$, but $5^{3} \equiv -1 \pmod{21}$, so the reduction is expected to fail in the sense that both gcds come out trivial ($1$ and $21$); the check below verifies this directly rather than calling `factor_attempt`, which is not required to handle that case specially. Do not edit this cell.

In [ ]:
assert order(2, 21) == 6, "order of 2 mod 21 should be 6"
r, g1, g2 = factor_attempt(2, 21)
assert {g1, g2} == {3, 7}, "the two gcds should be the prime factors 3 and 7"
assert order(5, 21) == 6, "order of 5 mod 21 should also be 6"
assert pow(5, 3, 21) == 20, "5**3 mod 21 should be 20, i.e. -1 mod 21"
assert gcd(pow(5, 3, 21) - 1, 21) == 1 and gcd(pow(5, 3, 21) + 1, 21) == 21, "a=5 should yield only trivial gcds"
print("A3 passed.")


## Part B: quantum networks and cryptography

Attempt after Lecture 8.

### Problem B1: entanglement swapping with a different resource

Nodes hold qubits $A$, $BC$, $D$. Pair $AB$ is in $\Psi^+$ and pair $CD$ is in $\Phi^+$. The middle node measures $BC$ in the Bell basis. Build the four-qubit state and the projection numerically.

Complete `swap_state()` to return $\Psi^+_{AB} \otimes \Phi^+_{CD}$ as an explicit 16-entry vector in the order $ABCD$, and `project_bc(state4, bell)` to return the (unnormalized) two-qubit state of $AD$ obtained by projecting qubits $B$ and $C$ onto a given Bell state `bell`.

In [ ]:
def swap_state():
    # TODO: return Psi+_{AB} tensor Phi+_{CD} as a 16-entry vector, qubits in order A, B, C, D.
    raise NotImplementedError


def project_bc(state4, bell):
    '''state4 is a 16-entry vector for qubits A,B,C,D. bell is a 4-entry Bell state for B,C.
    Return the unnormalized 4-entry state of A,D obtained by projecting B,C onto bell.'''
    # TODO: reshape state4 to (2,2,2,2) indexed (A,B,C,D) and contract the B,C indices against bell.
    raise NotImplementedError


Self-check. Projecting onto $\Phi^+$ should leave $AD$ proportional to $\Psi^+$ with probability $1/4$; projecting onto $\Psi^-$ should leave $AD$ proportional to $-\Phi^-$ with probability $1/4$. Do not edit this cell.

In [ ]:
state4 = swap_state()
ad_phip = project_bc(state4, Phip)
assert close(np.linalg.norm(ad_phip) ** 2, 0.25), "outcome Phi+ on BC should have probability 1/4"
assert close(ad_phip / np.linalg.norm(ad_phip), Psip), "AD should be left in Psi+"
ad_psim = project_bc(state4, Psim)
assert close(np.linalg.norm(ad_psim) ** 2, 0.25), "outcome Psi- on BC should have probability 1/4"
assert close(ad_psim / np.linalg.norm(ad_psim), -Phim), "AD should be left in -Phi-"
print("B1 passed.")


### Problem B2: BB84 with an intercept-resend eavesdropper

Lecture 8 shows that an eavesdropper who intercepts every qubit, measures it in a basis chosen uniformly at random, and resends what she found, causes a sifted-key error rate of $1/4$. Complete `bb84_intercept_resend_trial(rng)` so it simulates one position: Alice picks a uniformly random bit and basis, Eve picks a uniformly random basis and measures (collapsing the state, then resending what she found), Bob picks a uniformly random basis and measures. Return `None` if Alice's and Bob's bases differ (the position is not sifted), and otherwise return `True` if Bob's result matches Alice's bit and `False` if it does not.

In [ ]:
def bb84_intercept_resend_trial(rng):
    '''One simulated BB84 position with an intercept-resend eavesdropper. rng is a
    numpy.random.Generator. Returns None if the position is not sifted (Alice and Bob chose
    different bases), else True (Bob's result matches Alice's bit) or False.'''
    bases = {0: (k0, k1), 1: (plus, minus)}  # basis 0 = Z, basis 1 = X
    # TODO: implement the protocol described above.
    raise NotImplementedError


Self-check: over many trials, the sifted error rate (the fraction of non-`None` results that are `False`) should be close to $1/4$, matching Lecture 8. Do not edit this cell.

In [ ]:
rng = np.random.default_rng(2)
results = [bb84_intercept_resend_trial(rng) for _ in range(60000)]
sifted = [r for r in results if r is not None]
error_rate = 1 - sum(sifted) / len(sifted)
assert abs(error_rate - 0.25) < 0.02, f"sifted error rate {error_rate:.3f} should be close to 1/4"
print(f"B2 passed: sifted error rate {error_rate:.3f} over {len(sifted)} sifted positions.")


### Problem B3: what quantum computers threaten

This problem has no code. Answer in the markdown cell below, in two or three sentences per item: for RSA with a 2048-bit modulus, for AES-128, and for a one-time pad with a truly random key, state which quantum algorithm from the course is relevant (if any), whether a large quantum computer breaks the scheme, and what the effective security becomes.

**Your answer.**

*RSA-2048:*

*AES-128:*

*One-time pad:*

## Summary

Parts A and B are complete once every self-check cell above prints `passed` and Problem B3 is answered in its markdown cell. Submission arrangements are announced separately.